In [61]:
import pandas as pd
import numpy as np
df=pd.read_csv("Accidental_Drug_Related_Deaths.csv")

In [62]:
df.shape

(11981, 48)

In [63]:
missing_counts = df.isna().sum()
missing_percentage = (df.isna().mean()* 100).sort_values(ascending=False)
cols_over_60 = missing_percentage[missing_percentage > 60]

# Data Preprocessing

First we will standardize all y/n to 1's and 0's which is the data format that can be used in machine learning and analysis

In [64]:
drug_columns = [
    col for col in df.columns
    if col not in [
        "Date", "Date Type", "Age", "Sex", "Race", "Ethnicity",
        "Residence City", "Residence County", "Residence State","Injury City", "Injury County", "Injury State","Death City", "Death County", "Death State",
        "Injury Place", "Description of Injury","Location", "Location if Other","Cause of Death", "Manner of Death",
        "Other Significant Conditions","ResidenceCityGeo", "InjuryCityGeo", "DeathCityGeo",
    ]
]

df_clean=pd.read_csv("Accidental_Drug_Related_Deaths.csv")

for col in drug_columns:
    df_clean[col] = df_clean[col].apply(lambda x: 1 if x == "Y" else 0)

df_clean.to_csv("F:/0_TAYLORS/Y2S1/DM/assg1/df_clean0.csv", index=False)
df_clean


,Date,Date Type,Age,Sex,Race,Ethnicity,Residence City,Residence County,Residence State,Injury City,...,Xylazine,Gabapentin,Opiate NOS,Heroin/Morph/Codeine,Other Opioid,Any Opioid,Other,ResidenceCityGeo,InjuryCityGeo,DeathCityGeo
0,05/29/2012,Date of death,37.0,Male,Black,NaN,STAMFORD,FAIRFIELD,NaN,STAMFORD,...,0,0,0,0,0,0,0,"STAMFORD, CT\n(41.051924, -73.539475)","STAMFORD, CT\n(41.051924, -73.539475)","CT\n(41.575155, -72.738288)"
1,06/27/2012,Date of death,37.0,Male,White,NaN,NORWICH,NEW LONDON,NaN,NORWICH,...,0,0,0,0,0,0,0,"NORWICH, CT\n(41.524304, -72.075821)","NORWICH, CT\n(41.524304, -72.075821)","Norwich, CT\n(41.524304, -72.075821)"
2,03/24/2014,Date of death,28.0,Male,White,NaN,HEBRON,NaN,NaN,HEBRON,...,0,0,0,0,0,0,0,"HEBRON, CT\n(41.658069, -72.366324)","HEBRON, CT\n(41.658069, -72.366324)","Marlborough, CT\n(41.632043, -72.461309)"
3,12/31/2014,Date of death,26.0,Female,White,NaN,BALTIC,NaN,NaN,NaN,...,0,0,0,0,0,0,0,"BALTIC, CT\n(41.617221, -72.085031)","CT\n(41.575155, -72.738288)","Baltic, CT\n(41.617221, -72.085031)"
4,01/16/2016,Date of death,41.0,Male,White,NaN,SHELTON,FAIRFIELD,CT,SHELTON,...,0,0,0,0,0,1,0,"SHELTON, CT\n(41.316843, -73.092968)","SHELTON, CT\n(41.316843, -73.092968)","Bridgeport, CT\n(41.179195, -73.189476)"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11976,02/28/2023,Date of death,58.0,Female,White,"No, not Spanish/Hispanic/Latino",NEW HAVEN,NEW HAVEN,CT,NEW HAVEN,...,0,0,0,0,0,0,0,"NEW HAVEN, CT\n(41.3082517, -72.9241605)","NEW HAVEN, CT\n(41.3082517, -72.9241605)","CT\n(41.57350273, -72.738305908)"
11977,08/23/2023,Date of death,23.0,Male,White,"Yes, Mexican, Mexican American, Chicano",NEW HAVEN,NEW HAVEN,CT,NEW HAVEN,...,0,0,0,0,0,1,0,"NEW HAVEN, CT\n(41.3082517, -72.9241605)","NEW HAVEN, CT\n(41.3082517, -72.9241605)","CT\n(41.57350273, -72.738305908)"
11978,01/30/2023,Date of death,46.0,Male,White,"No, not Spanish/Hispanic/Latino",DANBURY,FAIRFIELD,CT,DANBURY,...,0,0,0,0,0,1,0,"DANBURY, CT\n(41.393666, -73.451539)","DANBURY, CT\n(41.393666, -73.451539)","CT\n(41.57350273, -72.738305908)"
11979,09/25/2023,Date of death,44.0,Male,White,"Yes, other Spanish/Hispanic/Latino",HARTFORD,HARTFORD,CT,HARTFORD,...,1,0,0,0,0,1,0,"HARTFORD, CT\n(41.765775, -72.673356)","HARTFORD, CT\n(41.765775, -72.673356)","CT\n(41.57350273, -72.738305908)"


Then we will run a variability report

In [65]:
# Heuristics
DOMINANCE_DROP_THRESHOLD = 0.995   # >= 99.5% same value among non-null -> drop as feature (usually)
HIGH_MISSING_THRESHOLD   = 0.90    # >= 90% missing -> usually drop/transform
LOW_UNIQUE_THRESHOLD     = 1       # 1 unique value among non-null -> constant


rows = []
n = len(df_clean)

for col in df_clean.columns:
    s = df_clean[col]

    non_null = int(s.notna().sum())
    missing_pct = float((n - non_null) / n) if n else np.nan

    # Unique values among non-null
    non_null_s = s.dropna()
    unique_non_null = int(non_null_s.nunique()) if non_null > 0 else 0

    # Most common value among non-null
    if non_null > 0:
        vc = non_null_s.value_counts(dropna=True)
        top_value = vc.index[0]
        top_count = int(vc.iloc[0])
        top_pct_of_non_null = float(top_count / non_null) if non_null else np.nan
    else:
        top_value = np.nan
        top_count = 0
        top_pct_of_non_null = np.nan

    # Recommendation logic (simple + explainable)
    recommendation = "Keep"
    reason = []

    if non_null == 0:
        recommendation = "Drop-as-feature"
        reason.append("All missing")
    else:
        if missing_pct >= HIGH_MISSING_THRESHOLD:
            recommendation = "Transform/Review"
            reason.append(f"High missingness ({missing_pct:.1%})")

        if unique_non_null <= LOW_UNIQUE_THRESHOLD:
            recommendation = "Drop-as-feature"
            reason.append("No variability (constant)")

        # Dominance check (near-constant)
        if pd.notna(top_pct_of_non_null) and top_pct_of_non_null >= DOMINANCE_DROP_THRESHOLD:
            # If it's already flagged as high missing, keep as Transform/Review,
            # otherwise drop as feature.
            if recommendation != "Transform/Review":
                recommendation = "Drop-as-feature"
            reason.append(f"Low variability (top value dominates {top_pct_of_non_null:.1%})")

    rows.append({
        "column": col,
        "non_null": non_null,
        "missing_%": round(missing_pct * 100, 2) if pd.notna(missing_pct) else np.nan,
        "unique_non_null": unique_non_null,
        "most_occuring_value": top_value,
        "most_occuring_count": top_count,
        "top_%_of_non_null": round(top_pct_of_non_null * 100, 2) if pd.notna(top_pct_of_non_null) else np.nan,
        "recommendation": recommendation,
        "reason": "; ".join(reason) if reason else ""
    })

report = pd.DataFrame(rows)

# Sort to surface "worst offenders" first (high missing, then high dominance)
report = report.sort_values(
    by=["missing_%", "top_%_of_non_null", "unique_non_null"],
    ascending=[False, False, True]
).reset_index(drop=True)

report.to_csv("F:/0_TAYLORS/Y2S1/DM/assg1/variability_report.csv", index=False)

print("\nTop 15 most problematic columns:")
print(report.head(15).to_string(index=False))


Top 15 most problematic columns:
               column  non_null  missing_%  unique_non_null                   most_occuring_value  most_occuring_count  top_%_of_non_null   recommendation                                       reason
    Location if Other      1194      90.03              543                    Friend's Residence                   68               5.70 Transform/Review                     High missingness (90.0%)
            Ethnicity      2565      78.59               13                              Hispanic                  972              37.89             Keep                                             
          Death State      6873      42.63                2                                    CT                 6872              99.99  Drop-as-feature Low variability (top value dominates 100.0%)
         Death County      8090      32.48               10                              HARTFORD                 2441              30.17             Keep            

Based on the report , I have decided to drop a few columns. This is because these columns provide little data as a feature , due to either high amounts of missingness , or low variability(indicated by unique_non_null)

In [ ]:
manner_counts = df["Manner of Death"].value_counts(dropna=False)
manner_percent = df["Manner of Death"].value_counts(normalize=True, dropna=False) * 100
manner_summary = pd.DataFrame({
    "Count": manner_counts,
    "Percentage (%)": manner_percent
})
df_clean.drop(columns=["Manner of Death","Other",], inplace=True)
df_clean.drop(columns=["Residence State","Injury State","Death State"], inplace=True)
manner_summary

,Count,Percentage (%)
Manner of Death,,
Accident,11942,99.674485
Pending,14,0.116852
accident,13,0.108505
NaN,9,0.075119
ACCIDENT,1,0.008347
Natural,1,0.008347
Acciddent,1,0.008347


Since there are already geo location under the columns residence city geo , injury city geo and death city geo , there is no need for the initial geo columns. 

analysis of location 